In [113]:
import pandas as pd
import numpy as np
from thefuzz import fuzz, process

In [114]:
url = "https://raw.githubusercontent.com/LiberoBiagi/ML_Nova_IMS_25-26/main/train.csv"

df = pd.read_csv("train.csv")
df.head()


,carID,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,69512,VW,Golf,2016.0,22290,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4.000000,0.0
1,53000,Toyota,Yaris,2019.0,13790,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1.000000,0.0
2,6366,Audi,Q2,2019.0,24990,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,56.0,4.000000,0.0
3,29021,Ford,FIESTA,2018.0,12500,anual,9102.0,Petrol,145.0,65.700000,1.0,50.0,-2.340306,0.0
4,10062,BMW,2 Series,2019.0,22995,Manual,1000.0,Petrol,145.0,42.800000,1.5,97.0,3.000000,0.0


# Data Exploration 

In [115]:
print(df.shape)

(75973, 14)


In [116]:
#df.carID.count() # No duplicates for CarID

In [117]:
df.describe()

,carID,year,price,mileage,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
count,75973.000000,74482.000000,75973.000000,74510.000000,68069.000000,68047.000000,74457.000000,74449.000000,74423.000000,74425.0
mean,37986.000000,2017.096611,16881.889553,23004.184088,120.329078,55.152666,1.660136,64.590667,1.994580,0.0
std,21931.660338,2.208704,9736.926322,22129.788366,65.521176,16.497837,0.573462,21.021065,1.472981,0.0
min,0.000000,1970.000000,450.000000,-58540.574478,-91.121630,-43.421768,-0.103493,1.638913,-2.345650,0.0
25%,18993.000000,2016.000000,10200.000000,7423.250000,125.000000,46.300000,1.200000,47.000000,1.000000,0.0
50%,37986.000000,2017.000000,14699.000000,17300.000000,145.000000,54.300000,1.600000,65.000000,2.000000,0.0
75%,56979.000000,2019.000000,20950.000000,32427.500000,145.000000,62.800000,2.000000,82.000000,3.000000,0.0
max,75972.000000,2024.121759,159999.000000,323000.000000,580.000000,470.800000,6.600000,125.594308,6.258371,0.0


In [118]:
df.dtypes

carID               int64
Brand              object
model              object
year              float64
price               int64
transmission       object
mileage           float64
fuelType           object
tax               float64
mpg               float64
engineSize        float64
paintQuality%     float64
previousOwners    float64
hasDamage         float64
dtype: object

In [119]:
df.isna().sum()

carID                0
Brand             1521
model             1517
year              1491
price                0
transmission      1522
mileage           1463
fuelType          1511
tax               7904
mpg               7926
engineSize        1516
paintQuality%     1524
previousOwners    1550
hasDamage         1548
dtype: int64

# Data Cleaning

## Cleaning text columns
### Resolving Spelling Issues in the Text columns
When exploring the data we see that there a multiple errors with the spelling of the Brand, model, transmission, fuelType columns. In the next section we will try to resolve that and create a coherent naming.

In [120]:
df["Brand"] = df["Brand"].str.lower().str.strip() # Convert brand names to lowercase and removes all beginning and trailing whitespace (e.g. space) from the column

df["model"] = df["model"].str.lower().str.strip()
df["transmission"] = df["transmission"].str.lower().str.strip()
df["fuelType"] = df["fuelType"].str.lower().str.strip()

df[["model", "transmission", "fuelType"]] = df[["model", "transmission", "fuelType"]].fillna("NaN") # Replace NaN with string NaN to avoid errors in the fuzzy algorithm (cant match NaNs)

In [121]:
# Optional display block, commented for compactnes
# Will show all the unqiue values for the text columns
"""print(df["Brand"].unique())
print("\n------------------------------------ \n")
print(df["model"].unique())
print("\n------------------------------------ \n")
print(df["transmission"].unique())
print("\n------------------------------------ \n")
print(df["fuelType"].unique())"""

'print(df["Brand"].unique())\nprint("\n------------------------------------ \n")\nprint(df["model"].unique())\nprint("\n------------------------------------ \n")\nprint(df["transmission"].unique())\nprint("\n------------------------------------ \n")\nprint(df["fuelType"].unique())'

#### Brands

We decided to do "manual" brand mapping because it gives the biggest controll factor while the number of values is managable. 
Also with some of the brand names beeing very short (e.g. vw) fuzzy algorithms would perform with reduced accuracy

In [122]:
brand_mapping = {
    "vw": "vw",
    "v": "vw",
    "w": "vw",
    
    "toyota": "toyota",
    "toyot": "toyota",
    "oyota": "toyota",
    
    "audi": "audi",
    "aud": "audi",
    "udi": "audi",
    "ud": "audi",
    
    "ford": "ford",
    "for": "ford",
    "ord": "ford",
    "or": "ford",
    
    "bmw": "bmw",
    "bm": "bmw",
    "mw": "bmw",
    
    "skoda": "skoda",
    "skod": "skoda",
    "koda": "skoda",
    "kod": "skoda",
    
    "opel": "opel",
    "ope": "opel",
    "pel": "opel",
    "pe": "opel",
    
    "mercedes": "mercedes",
    "mercede": "mercedes",
    "ercedes": "mercedes",
    "ercede": "mercedes",
    
    "hyundai": "hyundai",
    "hyunda": "hyundai",
    "yundai": "hyundai",
    "yunda": "hyundai"
}

df["Brand"] = df["Brand"].map(brand_mapping)

#### Similarity matching for Models, Transmission and fuel columns (Fuzzy Match)

In [123]:
models = ["golf", "veloste", "caddy", "yaris", "q2", "fiesta", "2 series", "3 series", "a3", "octavia", "passat", "focus", "insignia", "a class", "q3", "fabia", "ka+", "glc class", "i30", "c class", "polo", "e class", "q5", "up", "c-hr", "mokka x", "corsa", "astra", "tt", "5 series", "aygo", "4 series", "slk", "viva", "t-roc", "ecosport", "tucson", "x-class", "cl class", "ix20", "i20", "rapid", "a1", "auris", "sharan", "adam", "x3", "a8", "gls class", "b-max", "a4", "kona", "i10", "mokka", "s-max", "x2", "crossland x", "tiguan", "a5", "gle class", "zafira", "ioniq", "a6", "mondeo", "yeti outdoor", "x1", "scala", "s class", "1 series", "kamiq", "kuga", "tourneo connect", "q7", "gla class", "arteon", "sl class", "santa fe", "grandland x", "i800", "rav4", "touran", "citigo", "roomster", "prius", "corolla", "b class", "kodiaq", "v class", "caddy maxi life", "superb", "getz", "combo life", "beetle", "galaxy", "m3", "gtc", "x4", "ka", "ix35", "grand tourneo connect", "m4", "tourneo custom", "z4", "x5", "meriva", "rs6", "verso", "touareg", "shuttle", "cls class", "c-max", "puma", "cla class", "i40", "tiguan allspace", "6 series", "caravelle", "karoq", "i3", "grand c-max", "t-cross", "a7", "golf sv", "agila", "gt86", "yeti", "california", "land cruiser", "edge", "x6", "caddy life", "8 series", "fusion", "gl class", "scirocco", "z3", "proace verso", "hilux", "amarok", "cc", "7 series", "avensis", "eos", "m class", "grandland", "zafira tourer", "rs5", "r8", "mustang", "antara", "q8", "camry", "clk", "rs3", "jetta", "kadjar", "sq5", "rs4", "supra", "i8", "x7", "sq7", "g class", "s3", "crossland", "tigra", "escort", "glb class", "vivaro", "verso-s", "m5", "s4", "iq", "a2", "caddy maxi", "streetka", "cascada", "accent", "s8", "rs", "golf s", "ranger", "vectra", "ampera", "fox", "urban cruiser", "m2", "clc class", "m6", "s5", "terracan", "200", "220", "230", "NaN"]

short_models = [models[i] for i in range(len(models)) if len(models[i]) == 2]
short_models = list(set(short_models)) # get unique short model names as a list

transmission_types = ["semi-auto", "manual", "automatic", "unkown", "NaN", "other"]
fuel_types = ["petrol", "diesel", "hybrid", "electric", "other", "NaN"]

In [124]:
# Source for process.extractOne (fuzzy): https://github.com/seatgeek/thefuzz

for i in range(len(df)): 
    if len(df.model[i]) > 2: # Only perform fuzzy matching for models that have a name longer than 2 letters -> fuzzy will become fuzzy (unreliable) if names are to short
        df.loc[i, "model"] = process.extractOne(df.model[i], models)[0] # [0] because we get the name and score as a return -> score used for debugging
    elif len(df.model[i]) == 2: # Use the short names list for comparisons if the model names are 2 letters
        df.loc[i, "model"] = process.extractOne(df.model[i], short_models)[0]
    else: # We can define models with only one letter
        df.loc[i, "model"] = "NaN"

    df.loc[i, "transmission"] = process.extractOne(df.transmission[i], transmission_types)[0]
    df.loc[i, "fuelType"] = process.extractOne(df.fuelType[i], fuel_types)[0]

In [125]:
# Convert the str NaN values back to pd.NA for easier further processing and readability

df["model"] = df["model"].replace("NaN", pd.NA)
df["transmission"] = df["transmission"].replace(["unkown", "NaN", "other"], pd.NA)
df["fuelType"] = df["fuelType"].replace(["other", "NaN"], pd.NA)

#### Interpolate missing Brand names

In [126]:
brand_models = df.groupby("model")["Brand"].agg(lambda x: x.mode()) # Get the most frequent brand for each model -> returns df with model and brand
df = pd.merge(df, brand_models, on="model") # add the model and brand df to our main df (onyl add the brand columns, join on model)

df.drop('Brand_x', axis=1, inplace=True) # remove the old brand column
df = df.rename(columns={"Brand_y": "Brand"}) # rename new column
cols = ['carID', 'Brand', 'model', 'year', 'price', 'transmission', 'mileage', 'fuelType', 'tax', 'mpg', 'engineSize', 'paintQuality%', 'previousOwners', 'hasDamage'] # rearange column order
df = df[cols]

#### Interpolate missing transimission names
Five of the car models in our dataset where only produced with one transmission type but contained missing values in our dataset. For these models we can fill in the missing values as we know which transmission type it should be. This fixes the NA for 140 values.

In [127]:
#transmission_models = df.groupby("model")["transmission"].unique() #.agg(lambda x: x.mode())

"""adam: manual
camry: automatic 
ka: manual
m6: semi-auto
puma: manual"""

df.loc[df["model"]== "adam", "transmission"] = "manual"
df.loc[df["model"]== "camry", "transmission"] = "automatic"
df.loc[df["model"]== "ka", "transmission"] = "manual"
df.loc[df["model"]== "m6", "transmission"] = "semi-auto"
df.loc[df["model"]== "puma", "transmission"] = "manual"

## Cleaning numeric columns

In [128]:
df.year = df.year.round(0)
df.year =  pd.to_datetime(df["year"])
df.price = df.price.round(0) # Only integer prices, shouldn't change much
df.mileage = abs(df.mileage.round(0))
df.tax = abs(df.tax).round(0) # Turn negative values into positve
df.mpg = abs(df.mpg.round(1)) # Leave one after comma digit
df.previousOwners = abs(df.previousOwners.round())

### Removing unecessary comma digits
For most of the numerical columns we have entries with unnecessary after comma numbers. For example 2011.2108... we remove those after additional digits as we assume they are caused by errors and not important information. 
We do the same for negative values, if present. As we believe those are created by typos or system failures (e.g. taxes entered as negative number could mean the employee thought a negative number is required because taxes are deducted)

In [129]:
#df["paintQuality%"].sort_values().unique()

In [130]:
# Check number of of values impacted
# values < 4: 311 
# values > 100: 325
# Check if they are typos, based on mean values with and without wrong values -> no clear difference 
# Removing the values because of the low number of values impacted

In [131]:
#df2 = df.query("`paintQuality%` < 4 or `paintQuality%` > 100")
#df2.groupby("paintQuality%")[["price", "mileage"]].agg(["mean", "median", "count"]).round(2)

In [132]:
#df_clean = df.query("`paintQuality%` > 4 and `paintQuality%` < 100")
#df_clean.groupby("paintQuality%")[["price", "mileage"]].agg(["mean", "median", "count"]).round(2)

In [133]:
#df.shape[0]  -df2.shape[0]

In [134]:
# Filter the wrong values
df = df.query("`paintQuality%` > 4 and `paintQuality%` < 100").copy()

### Further numerical cleaning plan


### hasDamage column
This is a column that is filled by the customer prior to inspection. If the car has no damage the customer fills it in, the other values are left empty. Since it is impossible to determine the level of damage from all the other datapoints we decided to make changes to the column hasDamage. Mainly we change the purpose of the column to assesing if the customer stated that his car has no damage, in this case true (previous 0), else fales (previous missing).

In [135]:
df.hasDamage = df["hasDamage"].fillna(1) # We replace the missing values in hasDamage with 1 as missing values means the customer left the field empty
# Mean and Median very similar for both Damage values 
df.groupby("hasDamage")[["price", "year", "mileage", "paintQuality%"]].agg(["mean", "median"]).round(2).head(2)

price                                   year  \
               mean   median                          mean   
hasDamage                                                    
0.0        16872.13  14698.0 1970-01-01 00:00:00.000002017   
1.0        16900.02  14809.0 1970-01-01 00:00:00.000002017   

                                          mileage          paintQuality%  \
                                 median      mean   median          mean   
hasDamage                                                                  
0.0       1970-01-01 00:00:00.000002017  23444.90  17525.0         64.59   
1.0       1970-01-01 00:00:00.000002017  23324.71  16537.5         64.55   

                  
          median  
hasDamage         
0.0         65.0  
1.0         65.0

In [136]:
# Create the new column
df["stated_no_damage"] = ~df["hasDamage"].astype(bool)

In [137]:
df = df.drop(["hasDamage"], axis=1)

### Removing Columns -> move this after to interpolation as we maybe can use some of the values during interpolation

In [138]:
df.shape

(72047, 14)

In [139]:
# Remove values which we cant interpolate reliably
df = df[df.model.notna()]
df = df[df.year.notna()]
df = df[df.mileage.notna()]
df = df[df.previousOwners.notna()]

In [140]:
df.shape

(67853, 14)

In [141]:
# Remove values which could be interpolated with mode, but would increase bias in the data
df = df[df.transmission.notna()] # Missing values 2078
df = df[df.fuelType.notna()] # Missing values 1532
#df = df[df.engineSize.notna()] # Missing values 1411
df = df[df["paintQuality%"].notna()] # Missing values 1403

In [142]:
df.shape
# Share of data removed 14,19% 

(64398, 14)

### Tax Column
After comparing the mean and median tax statistics for the car models by year we decided to interpolate the missing values using the mean tax value grouped by model, year, transmission, fuel. We think this is a good approaximation of the expected tax amount of that specific car as the tax is based on emissions which vary depending on the factors defined in our grouping methode.

In [143]:
df.groupby(["model", "year", "transmission", "fuelType"])["tax"].agg(["mean", "median", "count"]).round(2).head(2)

mean  median  \
model    year                          transmission fuelType                  
1 series 1970-01-01 00:00:00.000002001 manual       petrol    125.0   125.0   
         1970-01-01 00:00:00.000002004 manual       diesel    200.0   200.0   

                                                              count  
model    year                          transmission fuelType         
1 series 1970-01-01 00:00:00.000002001 manual       petrol        1  
         1970-01-01 00:00:00.000002004 manual       diesel        1

In [144]:
df["tax"] = df["tax"].fillna(df.groupby(["model", "year", "transmission", "fuelType"])["tax"].transform("mean")).round(2)

### mpg Column

In [145]:
df.groupby(["model", "year", "transmission", "fuelType"])["mpg"].agg(["mean", "median", "count"]).round(2).head(2)

mean  median  \
model    year                          transmission fuelType                 
1 series 1970-01-01 00:00:00.000002001 manual       petrol    53.3    53.3   
         1970-01-01 00:00:00.000002004 manual       diesel    49.6    49.6   

                                                              count  
model    year                          transmission fuelType         
1 series 1970-01-01 00:00:00.000002001 manual       petrol        1  
         1970-01-01 00:00:00.000002004 manual       diesel        1

In [146]:
df["mpg"] = df["mpg"].fillna(df.groupby(["model", "year", "transmission", "fuelType"])["mpg"].transform("mean")).round(2)

## Engine Size

In [147]:
df["engineSize"] = df["mpg"].fillna(df.groupby(["model", "year", "transmission", "fuelType"])["mpg"].transform("mean")).round(2)

In [153]:
#df.isna().sum()

In [149]:
df = df[df.tax.notna()] # Remove values that couldn't be interpolated
df = df[df.mpg.notna()]

Brand -> Group by the model and Brand <br>
model -> remove (no clear identifcation thorugh: mpg, engineSize, year, transimission possible) <br>
year -> remove (no interpolation possible, car models where build accross multiple years) <br>
price -> no missing values <br>
transmission -> removed <br>
mileage -> remove (assuemd high volatility for years and mileage) <br>
fuelType ->Removed <br>
tax -> grouped by model, year, transmission, fule mean<br>
mpg -> grouped by model, year, transmission, fule mean <br>
engineSize -> grouped by model, year, transmission, fule mean <br>
paintQuality% -> TBD <br>
previousOwners -> Remove? <br>
hasDamage -> replace Missing values with 1<br>

In [150]:
# Todos
# Turn finished preprocessing into a function (seperate processing for training and test data)


In [151]:
df

,carID,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,stated_no_damage
0,69512,vw,golf,1970-01-01 00:00:00.000002016,22290,semi-auto,28421.0,petrol,95.95,11.4,11.4,63.0,4.0,True
1,53000,toyota,yaris,1970-01-01 00:00:00.000002019,13790,manual,4589.0,petrol,145.00,47.9,47.9,50.0,1.0,True
2,6366,audi,q2,1970-01-01 00:00:00.000002019,24990,semi-auto,3624.0,petrol,145.00,40.9,40.9,56.0,4.0,True
3,29021,ford,fiesta,1970-01-01 00:00:00.000002018,12500,manual,9102.0,petrol,145.00,65.7,65.7,50.0,2.0,True
4,10062,bmw,2 series,1970-01-01 00:00:00.000002019,22995,manual,1000.0,petrol,145.00,42.8,42.8,97.0,3.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74249,37194,mercedes,c class,1970-01-01 00:00:00.000002015,13498,manual,14480.0,petrol,125.00,53.3,53.3,78.0,0.0,True
74250,6265,audi,q3,1970-01-01 00:00:00.000002013,12495,semi-auto,52134.0,diesel,200.00,47.9,47.9,38.0,2.0,True
74251,54886,toyota,aygo,1970-01-01 00:00:00.000002017,8399,automatic,11304.0,petrol,145.00,67.0,67.0,57.0,3.0,True
74252,860,audi,q3,1970-01-01 00:00:00.000002015,12990,manual,69072.0,diesel,125.00,60.1,60.1,74.0,2.0,True
